In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
NUM_CLASSES = 3

model = models.resnet50(weights=None)  # no need to download weights again
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
model = model.to(device)

In [4]:
try:
    model.load_state_dict(torch.load("best_model_finetuned.pth", map_location=device))
    print("Loaded fine-tuned model (best_model_finetuned.pth)")
except FileNotFoundError:
    try:
        model.load_state_dict(torch.load("best_model.pth", map_location=device))
        print("Loaded base model (best_model.pth)")
    except FileNotFoundError:
        raise FileNotFoundError("No trained model found. Please run training first to generate best_model.pth or best_model_finetuned.pth")

model.eval()

Loaded fine-tuned model (best_model_finetuned.pth)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [5]:
IMG_SIZE = 224

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet std
    )
])

In [6]:
class_names = ['normal', 'osteopenia', 'osteoporosis']

In [7]:
import cv2
import numpy as np
from PIL import Image, ImageStat

def diagnose_image(img_path):
    img = Image.open(img_path).convert("RGB")
    gray = np.array(img.convert('L'))
    
    # Pixel intensity statistics
    pixel_array = np.array(img)
    
    # Color channel statistics
    r_mean, g_mean, b_mean = np.mean(pixel_array[:,:,0]), np.mean(pixel_array[:,:,1]), np.mean(pixel_array[:,:,2])
    r_std, g_std, b_std = np.std(pixel_array[:,:,0]), np.std(pixel_array[:,:,1]), np.std(pixel_array[:,:,2])
    
    # Grayscale statistics
    gray_mean, gray_std = np.mean(gray), np.std(gray)
    
    # Histogram analysis
    hist, bins = np.histogram(gray.flatten(), bins=256, range=[0,256])
    
    print(f"Image: {img_path}")
    print(f"  RGB means - R:{r_mean:.1f}, G:{g_mean:.1f}, B:{b_mean:.1f}")
    print(f"  RGB stds - R:{r_std:.1f}, G:{g_std:.1f}, B:{b_std:.1f}")
    print(f"  Grayscale - Mean:{gray_mean:.1f}, Std:{gray_std:.1f}")
    print(f"  Dynamic range: {gray.min()}-{gray.max()}")
    print(f"  Histogram peaks: {np.argsort(hist)[-5:][::-1]} (top 5 intensity bins)\n")

def is_likely_xray(image):
    """
    Enhanced X-ray detection based on pixel-wise coloration and intensity characteristics.
    More lenient intensity ranges to accommodate various X-ray types.
    """
    pixel_array = np.array(image)
    gray = np.array(image.convert('L'))
    
    # 1. Color channel correlation - X-rays should have very similar RGB values
    r, g, b = pixel_array[:,:,0], pixel_array[:,:,1], pixel_array[:,:,2]
    rg_correlation = np.corrcoef(r.flatten(), g.flatten())[0,1]
    rb_correlation = np.corrcoef(r.flatten(), b.flatten())[0,1]
    gb_correlation = np.corrcoef(g.flatten(), b.flatten())[0,1]
    
    avg_correlation = (rg_correlation + rb_correlation + gb_correlation) / 3
    
    if avg_correlation < 0.80:  # Slightly more lenient
        print(f"❌ Rejected: Low RGB correlation ({avg_correlation:.3f} < 0.80)")
        return False
    
    # 2. Color channel standard deviations - should be very similar for X-rays
    r_std, g_std, b_std = np.std(r), np.std(g), np.std(b)
    max_std_diff = max(abs(r_std - g_std), abs(r_std - b_std), abs(g_std - b_std))
    
    if max_std_diff > 12:  # More lenient
        print(f"❌ Rejected: High RGB std difference ({max_std_diff:.2f} > 12)")
        return False
    
    # 3. Overall color saturation - X-rays should be mostly grayscale
    rgb_means = [np.mean(r), np.mean(g), np.mean(b)]
    max_rgb_diff = max(rgb_means) - min(rgb_means)
    
    if max_rgb_diff > 20:  # More lenient for slight color casts
        print(f"❌ Rejected: High RGB mean difference ({max_rgb_diff:.2f} > 20)")
        return False
    
    # 4. Intensity distribution - Much more lenient ranges for various X-ray types
    gray_mean, gray_std = np.mean(gray), np.std(gray)
    
    # X-rays can be much darker (especially bone density X-rays) or brighter
    if gray_mean < 20 or gray_mean > 220:  # Expanded range
        print(f"❌ Rejected: Extreme mean intensity ({gray_mean:.1f} not in [20, 220])")
        return False
    
    # X-rays can have various contrast levels
    if gray_std < 10 or gray_std > 100:  # More lenient
        print(f"❌ Rejected: Poor intensity distribution (std: {gray_std:.1f} not in [10, 100])")
        return False
    
    # 5. Dynamic range check - More lenient
    dynamic_range = gray.max() - gray.min()
    if dynamic_range < 30:  # Lower threshold
        print(f"❌ Rejected: Low dynamic range ({dynamic_range} < 30)")
        return False
    
    print(f"✅ X-ray characteristics detected:")
    print(f"   RGB correlation: {avg_correlation:.3f}")
    print(f"   RGB std difference: {max_std_diff:.2f}")
    print(f"   RGB mean difference: {max_rgb_diff:.2f}")
    print(f"   Intensity: mean={gray_mean:.1f}, std={gray_std:.1f}, range={dynamic_range}")
    
    return True

In [8]:
def predict_image(img_path, confidence_threshold=0.5):
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        return f"❌ Error loading image: {str(e)}"
    
    print(f"Analyzing image: {img_path}")
    
    # First check if it's likely an X-ray
    if not is_likely_xray(image):
        return "❌ Rejected: Image does not appear to be an X-ray. Please upload a valid medical X-ray image."

    # Normal prediction for confirmed X-rays
    img_tensor = val_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        max_prob, pred_idx = torch.max(probabilities, 0)
        predicted_class = class_names[pred_idx.item()]
        confidence = max_prob.item()
        
        # Get all probabilities for better context
        probs_dict = {class_names[i]: probabilities[i].item() for i in range(len(class_names))}
        second_highest = sorted(probs_dict.values())[-2]

    # More lenient confidence handling
    if confidence < confidence_threshold:
        if confidence < 0.3:
            return f"❌ Very low confidence ({confidence:.2f}) – image quality may be poor or doesn't contain clear bone density information."
        else:
            # For moderate confidence, still provide prediction with warning
            return f"⚠️ Low confidence prediction: {predicted_class} (confidence: {confidence:.2f}) - Consider using a clearer X-ray image for better accuracy."

    # Add warning for moderate confidence but still accept
    if confidence < 0.65:
        return f"✅ Prediction: {predicted_class} (confidence: {confidence:.2f}) - Moderate confidence, results should be verified."

    return f"✅ Prediction: {predicted_class} (confidence: {confidence:.2f})"

In [9]:
# Test with the current image
result = predict_image("spine1.jpg")
print(f"Prediction: {result}")

Prediction: ❌ Error loading image: [Errno 2] No such file or directory: 'spine1.jpg'
